<a href="https://colab.research.google.com/github/Hwk040319/MJY-ML/blob/main/00_quickstart_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 배터리 열폭주 이미지 분류 · Colab 퀵스타트

## 0. GPU 확인

In [1]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
# False 면 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 이 셀 다시 실행

GPU 사용 가능: True


## 1. 코드 내려받기

In [2]:
!git clone https://github.com/Hwk040319/MJY-ML.git
%cd MJY-ML
!pip install -q -r requirements.txt

Cloning into 'MJY-ML'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 131 (delta 72), reused 31 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 63.22 KiB | 851.00 KiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/MJY-ML


## 2. 데이터 내려받기



In [3]:
!pip install -q gdown

# 1회차 강의 실습용 (약 700장, 22MB)
FILE_ID = '13DWY5tg_L4SYxujkdVQ89qEQZC08lr7L'

!gdown "https://drive.google.com/uc?id=$FILE_ID" -O data.tar
!mkdir -p data && tar -xf data.tar -C data
!ls data

Downloading...
From: https://drive.google.com/uc?id=13DWY5tg_L4SYxujkdVQ89qEQZC08lr7L
To: /content/MJY-ML/data.tar
100% 22.1M/22.1M [00:00<00:00, 48.8MB/s]
public_val  train


In [5]:
# Validation의 실험 1개를 Train으로 이동해
# Train 이미지가 Validation 이미지보다 많도록 조정합니다.
#
# 주의:
# 같은 실험에서 촬영된 연속 이미지는 서로 매우 비슷하므로
# 이미지를 개별적으로 섞지 않고 experiment_id 단위로 이동합니다.

from pathlib import Path
import shutil
import pandas as pd


data_root = Path('data')
train_dir = data_root / 'train'
val_dir = data_root / 'public_val'

train_labels_path = train_dir / 'labels.csv'
val_labels_path = val_dir / 'labels.csv'

train_df = pd.read_csv(train_labels_path)
val_df = pd.read_csv(val_labels_path)


# Train에 남길 목표 experiment_id 개수입니다.
#
# 6개: Train 513장 / Valid 180장 — 약 74:26
desired_train_experiments = 6


current_train_experiments = train_df['experiment_id'].nunique()
n_experiments_to_move = (
    desired_train_experiments - current_train_experiments
)


if n_experiments_to_move <= 0:
    print(
        "이미 Train 실험 수가 목표 이상입니다:",
        current_train_experiments
    )

else:
    # 결과가 실행할 때마다 달라지지 않도록
    # experiment_id를 정렬한 뒤 앞에서부터 선택합니다.
    val_experiment_ids = sorted(
        val_df['experiment_id'].unique()
    )

    move_experiment_ids = val_experiment_ids[
        :n_experiments_to_move
    ]

    if len(move_experiment_ids) < n_experiments_to_move:
        raise RuntimeError(
            "Validation에 이동할 실험이 충분하지 않습니다."
        )

    # 이동할 실험에 포함된 라벨 행을 선택합니다.
    rows_to_move = val_df[
        val_df['experiment_id'].isin(move_experiment_ids)
    ].copy()

    # 선택된 실험의 실제 이미지 파일을
    # public_val/images에서 train/images로 이동합니다.
    for image_name in rows_to_move['image_name']:
        source = val_dir / 'images' / image_name
        destination = train_dir / 'images' / image_name

        if not source.is_file():
            raise FileNotFoundError(
                f"이동할 이미지가 없습니다: {source}"
            )

        if destination.exists():
            raise FileExistsError(
                f"Train에 같은 파일명이 이미 있습니다: {destination}"
            )

        shutil.move(source, destination)

    # 이동한 행은 Train 라벨에 추가합니다.
    train_df = pd.concat(
        [train_df, rows_to_move],
        ignore_index=True
    )

    # 이동한 행은 Validation 라벨에서 제거합니다.
    val_df = val_df[
        ~val_df['experiment_id'].isin(move_experiment_ids)
    ].reset_index(drop=True)

    # 변경된 labels.csv를 저장합니다.
    train_df.to_csv(train_labels_path, index=False)
    val_df.to_csv(val_labels_path, index=False)

    print("Train으로 이동한 experiment_id:")
    print(move_experiment_ids)


# 변경 결과를 확인합니다.
print()
print(
    f"Train: {len(train_df)}장 / "
    f"실험 {train_df['experiment_id'].nunique()}개"
)
print(
    f"Valid: {len(val_df)}장 / "
    f"실험 {val_df['experiment_id'].nunique()}개"
)

print()
print("Train 클래스 분포:")
print(train_df['target'].value_counts().sort_index())

print()
print("Valid 클래스 분포:")
print(val_df['target'].value_counts().sort_index())

이미 Train 실험 수가 목표 이상입니다: 5

Train: 423장 / 실험 5개
Valid: 270장 / 실험 3개

Train 클래스 분포:
target
0    150
1    150
2    123
Name: count, dtype: int64

Valid 클래스 분포:
target
0    90
1    90
2    90
Name: count, dtype: int64


## 3. 데이터 검사




In [6]:
!python check_data.py --data-root data

데이터 검사 시작: /content/MJY-ML/data
------------------------------------------------------------
[train] 이미지 423장 / 라벨 423행
         초기    150 ( 35.5%)  중기    150 ( 35.5%)  후기    123 ( 29.1%)
         experiment_id 5개
[public_val] 이미지 270장 / 라벨 270행
         초기     90 ( 33.3%)  중기     90 ( 33.3%)  후기     90 ( 33.3%)
         experiment_id 3개
[private_test] 폴더 없음 (정상 - 참가자에게 배포되지 않는 분할입니다)
------------------------------------------------------------
문제 없음. train_baseline.py 를 실행해도 좋습니다.


## 3.5 데이터 구조와 전처리 직접 확인 (선택)

채점이나 제출과 무관한 확인용 셀입니다. `train_baseline.py`를 실행하기 전에, 실제로 어떤 이미지가 어떤 라벨로 들어가는지, 그리고 5단계 중 Resize·Tensor 변환·정규화를 코드로 직접 확인합니다.

In [7]:
# labels.csv 구조 확인 — 이미지 파일명과 라벨이 어떻게 짝지어져 있는지
import pandas as pd

df = pd.read_csv('data/train/labels.csv')
print('열 구성:', list(df.columns))          # image_name, target, original_stage, experiment_id
print('행 개수:', len(df))
df.head()

열 구성: ['image_name', 'target', 'original_stage', 'experiment_id', 'source_partition', 'source_label_archive']
행 개수: 423


,image_name,target,original_stage,experiment_id,source_partition,source_label_archive
0,20250901_152742_1.png,0,1,20250901003,Training,TL_센서_각형_280_100_과충전_20250901_20250901003.zip
1,20250901_152901_1.png,0,1,20250901003,Training,TL_센서_각형_280_100_과충전_20250901_20250901003.zip
2,20250901_152745_1.png,0,1,20250901003,Training,TL_센서_각형_280_100_과충전_20250901_20250901003.zip
3,20250901_151930_1.png,0,1,20250901003,Training,TL_센서_각형_280_100_과충전_20250901_20250901003.zip
4,20250901_152915_1.png,0,1,20250901003,Training,TL_센서_각형_280_100_과충전_20250901_20250901003.zip


In [8]:
# Colab 한글 폰트 설치 — 런타임마다 한 번 실행
!apt-get update -qq
!apt-get install -y -qq fonts-nanum > /dev/null

import matplotlib as mpl
import matplotlib.font_manager as fm
from pathlib import Path

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
assert Path(font_path).is_file(), f"폰트를 찾을 수 없습니다: {font_path}"

# 런타임을 재시작하지 않아도 현재 세션에 바로 등록
fm.fontManager.addfont(font_path)

mpl.rcParams['font.family'] = 'NanumGothic'
mpl.rcParams['axes.unicode_minus'] = False

print("한글 폰트 설정 완료:", font_path)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
한글 폰트 설정 완료: /usr/share/fonts/truetype/nanum/NanumGothic.ttf


In [9]:
# 클래스(초기/중기/후기)별로 대표 이미지를 한 장씩 확인합니다.
# 모델 학습 전에 이미지와 라벨이 올바르게 연결되어 있는지
# 사람이 직접 눈으로 확인하기 위한 코드입니다.

from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd


# 이미지 파일명, 분류 라벨, 원본 단계 등이 저장된 CSV를 불러옵니다.
df = pd.read_csv('data/train/labels.csv')

# target 숫자가 의미하는 클래스 이름입니다.
# target=0: 초기, target=1: 중기, target=2: 후기
CLASS_NAMES = ['초기', '중기', '후기']


# 초기·중기·후기 이미지를 가로로 한 장씩 표시할 공간을 만듭니다.
# 1행 3열이므로 총 3개의 그래프 영역이 만들어집니다.
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))


# cls에 0, 1, 2가 순서대로 들어갑니다.
# 따라서 이미지도 초기 → 중기 → 후기 순서로 표시됩니다.
for cls in range(3):

    # 현재 클래스에 해당하는 행들 중 첫 번째 행을 대표 샘플로 선택합니다.
    row = df[df['target'] == cls].iloc[0]

    # labels.csv에 기록된 이미지 파일명을 이용해 실제 이미지를 불러옵니다.
    # convert('RGB')는 이미지를 빨강·초록·파랑의 3채널 형식으로 통일합니다.
    image_path = f"data/train/images/{row['image_name']}"
    img = Image.open(image_path).convert('RGB')

    # 현재 클래스에 해당하는 그래프 영역에 이미지를 표시합니다.
    axes[cls].imshow(img)

    # 이미지 위에 클래스 이름, target 번호, 원본 단계를 표시합니다.
    # 파일명은 화면이 복잡해지지 않도록 표시하지 않습니다.
    axes[cls].set_title(
        f"{CLASS_NAMES[cls]}\n"
        f"(target={row['target']}, 원본 단계={row['original_stage']})",
        fontsize=12,
        pad=8
    )

    # 이미지 주변의 좌표축과 눈금은 필요하지 않으므로 숨깁니다.
    axes[cls].axis('off')


# 세 이미지 전체를 설명하는 제목을 그림 위쪽에 표시합니다.
fig.suptitle(
    "클래스별 샘플 이미지 — 초기 · 중기 · 후기",
    fontsize=16,
    y=0.98
)


# 전체 제목과 각 이미지 제목이 서로 겹치지 않도록 여백을 조정합니다.
fig.subplots_adjust(
    left=0.03,     # 그림 왼쪽 여백
    right=0.97,    # 그림 오른쪽 여백
    bottom=0.03,   # 그림 아래쪽 여백
    top=0.76,      # 전체 제목을 위한 위쪽 공간
    wspace=0.15    # 이미지 사이의 가로 간격
)


# 완성된 결과를 화면에 출력합니다.
plt.show()

<Figure size 1300x450 with 3 Axes>

In [10]:
# 전처리 흐름(Resize -> ToTensor -> Normalize)을 한 단계씩 직접 실행
from torchvision import transforms
from common import IMAGENET_MEAN, IMAGENET_STD, get_transforms

sample_name = df.iloc[0]['image_name']
sample_img = Image.open(f"data/train/images/{sample_name}").convert('RGB')
print('0) 원본 크기:', sample_img.size)

step1 = transforms.Resize((224, 224))(sample_img)
print('1) Resize 후 크기:', step1.size)                      # (224, 224)로 통일

step2 = transforms.ToTensor()(step1)
print('2) ToTensor 후 shape:', tuple(step2.shape),
      '값 범위:', round(step2.min().item(), 3), '~', round(step2.max().item(), 3))
# [3, 224, 224], 0~255 픽셀값을 255로 나눠 0~1 범위로 변환

step3 = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(step2)
print('3) Normalize 후 값 범위:', round(step3.min().item(), 2), '~', round(step3.max().item(), 2))
# ImageNet 평균/표준편차로 재조정 -> 사전학습된 ResNet18이 기대하는 입력 분포에 맞춤

final = get_transforms(224, train=False)(sample_img)
print('get_transforms() 결과와 동일한가:', torch.allclose(final, step3))  # True면 위 세 단계와 같은 전처리

0) 원본 크기: (336, 297)
1) Resize 후 크기: (224, 224)
2) ToTensor 후 shape: (3, 224, 224) 값 범위: 0.0 ~ 0.988
3) Normalize 후 값 범위: -2.12 ~ 2.59
get_transforms() 결과와 동일한가: True


## 4. Baseline 학습

In [11]:
!python train_baseline.py \
  --data-root data \
  --output-dir outputs/baseline \
  --epochs 5 \
  --batch-size 32 \
  --lr 1e-3

데이터 검사 시작: /content/MJY-ML/data
------------------------------------------------------------
[train] 이미지 423장 / 라벨 423행
         초기    150 ( 35.5%)  중기    150 ( 35.5%)  후기    123 ( 29.1%)
         experiment_id 5개
[public_val] 이미지 270장 / 라벨 270행
         초기     90 ( 33.3%)  중기     90 ( 33.3%)  후기     90 ( 33.3%)
         experiment_id 3개
[private_test] 폴더 없음 (정상 - 참가자에게 배포되지 않는 분할입니다)
------------------------------------------------------------
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training 423장  {'초기': 150, '중기': 150, '후기': 123}
Public Validation 270장


## 5. 결과 확인

## 이미지 한 장 예측 (선택)

In [12]:
from pathlib import Path
from PIL import Image
from IPython.display import display

# public_val 이미지 중 한 장을 선택합니다.
sample_image = next(Path("data/public_val/images").glob("*.png"))

# 선택한 이미지를 화면에 표시합니다.
display(Image.open(sample_image))

# 표시된 이미지를 학습된 모델로 예측합니다.
!python predict_one.py \
    --image "$sample_image" \
    --checkpoint outputs/baseline/best_model.pt

<PIL.PngImagePlugin.PngImageFile image mode=RGB size=151x151>

{
  "image": "data/public_val/images/20250911_151902_1.png",
  "predicted_class": 2,
  "predicted_label": "후기",
  "probabilities": {
    "초기": 0.037777,
    "중기": 0.29963,
    "후기": 0.662593
  }
}


## 학습 지표 확인

In [16]:
# 이 셀만 실행해도 그래프가 나오도록 필요한 결과를 먼저 불러옵니다.
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

# 위의 기본 학습 셀에서 --output-dir로 지정한 폴더입니다.
output_dir = Path('outputs/baseline')
report_path = output_dir / 'validation_report.json'
learning_curves_path = output_dir / 'learning_curves.png'
confusion_matrix_path = output_dir / 'confusion_matrix.png'

# 학습이 완료되지 않았거나 output_dir이 다르면 이해하기 쉬운 오류를 표시합니다.
for result_path in [report_path, learning_curves_path, confusion_matrix_path]:
    if not result_path.is_file():
        raise FileNotFoundError(
            f"결과 파일을 찾을 수 없습니다: {result_path}\n"
            "먼저 모델 학습 셀을 실행하고 output_dir 경로를 확인하세요."
        )

with open(report_path, encoding='utf-8') as file:
    report = json.load(file)

learning_curves = np.asarray(PILImage.open(learning_curves_path).convert('RGB'))
confusion_matrix_image = np.asarray(
    PILImage.open(confusion_matrix_path).convert('RGB')
)

# 혼동행렬의 각 행은 실제 클래스, 각 열은 예측 클래스를 뜻합니다.
CLASS_NAMES = ['초기', '중기', '후기']
cm = np.asarray(report['confusion_matrix'], dtype=float)
row_totals = cm.sum(axis=1)
class_recall = np.divide(
    np.diag(cm),
    row_totals,
    out=np.zeros_like(row_totals),
    where=row_totals != 0,
)
macro_recall = float(class_recall.mean())

print("Accuracy:", round(report['accuracy'], 4))
print("Macro F1:", round(report['macro_f1'], 4))
print("Macro Recall:", round(macro_recall, 4))

# 세 그래프를 작게 만들어 가로로 나란히 표시합니다.
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4),
    gridspec_kw={
        'width_ratios': [1.7, 1, 1]
    }
)


# 1. 학습 곡선
axes[0].imshow(learning_curves)
axes[0].axis('off')


# 2. Confusion Matrix
axes[1].imshow(confusion_matrix_image)
axes[1].axis('off')


# 3. 클래스별 Recall
recall_bars = axes[2].bar(
    CLASS_NAMES,
    class_recall,
    color=[
        '#5B8FF9',
        '#61DDAA',
        '#F6BD16'
    ]
)

# 막대 위에 Recall 값을 표시합니다.
axes[2].bar_label(
    recall_bars,
    labels=[
        f"{value:.3f}"
        for value in class_recall
    ],
    padding=3
)

axes[2].set_ylim(0, 1.08)
axes[2].set_xlabel("배터리 상태")
axes[2].set_ylabel('Recall')
axes[2].set_title(
    f"클래스별 Recall\nMacro Recall: {macro_recall:.4f}"
)
axes[2].grid(
    axis='y',
    alpha=0.25
)

plt.tight_layout()
plt.show()

<Figure size 1500x400 with 3 Axes>

## 6. 개선 실험(common.py, train_baseline.py 참고)


In [1]:
# A. 데이터 증강
!python train_baseline.py --data-root data --augment --epochs 5 --output-dir outputs/exp_aug
# --augment를 추가하면 학습 이미지를 매번 조금씩 변화하면서 사용, ex) 이미지 일부를 무작위로 확대·잘라내기, 50% 확률로 좌우 반전, 최대 ±10도 회전, 밝기 변화, 대비 변화

# B. 클래스 가중치
!python train_baseline.py --data-root data --use-class-weights --epochs 5 --output-dir outputs/exp_weight
# --use-class-weights는 데이터가 적은 클래스에서 틀렸을 때 더 큰 손실을 부여. 즉 데이터가 적은 클래스에 더 많은 의미를 부여.
# 현재 코드에서는 가중치 = 전체 이미지 수 ÷ (클래스 수 × 해당 클래스 이미지 수) > 초기: 0.59, 중기:0.99, 후기: 3.22 > 후기 클래스를 잘 못 분류했을 때 초기에 비해 5.6배 더 크게 의미를 부여함.

# C1. 미세조정 대조군: backbone 고정, lr만 1e-4로 변경
!python train_baseline.py --data-root data --lr 1e-4 --epochs 5 --output-dir outputs/exp_lr_control
# 학습률을 작게: 보다 안전하게 학습, but epochs가 작으면 제대로 학습이 되지 않을 수 있음

# C2. 전체 미세조정: C1과 같은 lr에서 unfreeze만 추가
!python train_baseline.py --data-root data --unfreeze --lr 1e-4 --epochs 5 --output-dir outputs/exp_ft
# Backbone: 이미지의 특징을 추출하는 CNN 부분, Head: 추출된 특징으로 초기·중기·후기를 분류하는 마지막 층
# --unfreeze를 넣으면 Backbone 고정을 해제해서 ResNet18 전체를 학습.
# 왜 학습률을 낮추는가? ResNet18은 이미 ImageNet에서 유용한 특징을 학습한 상태 > 큰 학습률로 전체 모델을 수정하면 기존 특징이 빠르게 망가질 수 있음. 그래서 전체 미세조정에서는 상대적으로 작은 1e-4를 사용.


# D. 옵티마이저 변경 (adamw(기본) > sgd + momentum(더 빠르고 날카롭게 떨어짐))
!python train_baseline.py --data-root data --optimizer sgd --epochs 5 --output-dir outputs/exp_sgd

python3: can't open file '/content/train_baseline.py': [Errno 2] No such file or directory
python3: can't open file '/content/train_baseline.py': [Errno 2] No such file or directory
python3: can't open file '/content/train_baseline.py': [Errno 2] No such file or directory
python3: can't open file '/content/train_baseline.py': [Errno 2] No such file or directory
python3: can't open file '/content/train_baseline.py': [Errno 2] No such file or directory
